# AttentionBench — Explore & Evaluate

**Can models both filter distractions AND maintain focus over extended tasks?**

Two dimensions of attention, tested independently:
1. **Signal-in-Noise Titration** — selective attention (filtering noise)
2. **Vigilance Decrement** — sustained attention (focus over repetition)

Works on **Kaggle**, **Google Colab**, and **locally**.

In [ ]:
# Environment setup — works on Kaggle, Colab, and local
import os, sys, subprocess

ON_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    # Clone repo and install deps on Colab
    if not os.path.exists("attention-bench"):
        subprocess.run(["git", "clone", "https://github.com/42euge/attention-bench.git"], check=False)
    os.chdir("attention-bench")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "matplotlib"], check=True)

# Add src to path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if not ON_COLAB else os.getcwd()
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

# Generate dataset if not present
DATA_DIR = os.path.join(REPO_ROOT, "data")
if not os.path.exists(os.path.join(DATA_DIR, "signal_in_noise.json")):
    print("Generating dataset...")
    subprocess.run([sys.executable, os.path.join(REPO_ROOT, "src", "generate.py")], check=True)

print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Local'}")
print(f"Data dir: {DATA_DIR}")

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

# Load datasets
with open(os.path.join(DATA_DIR, "signal_in_noise.json")) as f:
    sin_raw = json.load(f)
with open(os.path.join(DATA_DIR, "vigilance.json")) as f:
    vig_raw = json.load(f)
with open(os.path.join(DATA_DIR, "manifest.json")) as f:
    manifest = json.load(f)

sin_df = pd.DataFrame(sin_raw)
vig_df = pd.DataFrame(vig_raw)

print(f"AttentionBench v{manifest['version']}")
print(f"  Signal-in-Noise: {len(sin_df)} items")
print(f"  Vigilance:       {len(vig_df)} items")
print(f"  Total:           {manifest['total_items']} items")

## 1. Signal-in-Noise: Dataset Structure

Each item embeds a ~200-word passage with 5 factual questions inside increasing amounts of noise.
The key variable is the **noise ratio** — how many words of noise per word of signal.

In [ ]:
# Prompt sizes by noise ratio and type
sin_df["prompt_len"] = sin_df["prompt"].str.len()
sin_df["prompt_words"] = sin_df["prompt"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Prompt length in characters
pivot_chars = sin_df.pivot_table(index="noise_ratio", columns="noise_type", values="prompt_len", aggfunc="mean")
pivot_chars.plot(kind="bar", ax=axes[0], rot=0)
axes[0].set_title("Mean prompt length (characters)")
axes[0].set_xlabel("Noise ratio (noise:signal)")
axes[0].set_ylabel("Characters")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
axes[0].legend(title="Noise type")

# Prompt length in words
pivot_words = sin_df.pivot_table(index="noise_ratio", columns="noise_type", values="prompt_words", aggfunc="mean")
pivot_words.plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Mean prompt length (words)")
axes[1].set_xlabel("Noise ratio (noise:signal)")
axes[1].set_ylabel("Words")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
axes[1].legend(title="Noise type")

plt.tight_layout()
plt.show()

# Summary table
print(pivot_words.round(0).astype(int).to_string())

In [ ]:
# Coverage: passages × noise types × ratios
print("Passages:", sin_df["passage_id"].nunique())
print("Domains:", sin_df["domain"].unique().tolist())
print()
print("Items per configuration:")
print(sin_df.groupby(["noise_type", "noise_ratio"]).size().unstack(fill_value=0).to_string())

In [ ]:
# Sample prompt at low noise (1:1) — the signal is clearly visible
sample_low = sin_df[(sin_df["noise_ratio"] == 1) & (sin_df["passage_id"] == "passage_01") & (sin_df["noise_type"] == "unrelated")].iloc[0]
print(f"=== {sample_low['task_id']} ({sample_low['prompt_words']} words) ===\n")
print(sample_low["prompt"][:2000])
print(f"\n... [{sample_low['prompt_words']} words total]")
print(f"\nExpected answers: {sample_low['answers']}")

In [ ]:
# Sample adversarial prompt at high noise (25:1) — signal buried in misleading content
sample_high = sin_df[(sin_df["noise_ratio"] == 25) & (sin_df["passage_id"] == "passage_01") & (sin_df["noise_type"] == "adversarial")].iloc[0]
print(f"=== {sample_high['task_id']} ({sample_high['prompt_words']} words) ===\n")
print(sample_high["prompt"][:3000])
print(f"\n... [{sample_high['prompt_words']} words total, showing first 3000 chars]")
print(f"\nExpected answers: {sample_high['answers']}")

## 2. Vigilance Decrement: Dataset Structure

Each item contains 100 identical, trivially easy sub-tasks. We measure whether accuracy decays over position.
Oddball variants insert one unexpected task type to test inattentional blindness.

In [ ]:
# Vigilance items overview
for _, row in vig_df.iterrows():
    oddball = f"oddball at position {row['oddball_position']}" if row["oddball_position"] else "no oddball"
    prompt_words = len(row["prompt"].split())
    print(f"{row['task_id']}: {row['num_subtasks']} subtasks, {oddball}, {prompt_words} words")

In [ ]:
# Sample vigilance prompt (first 30 items of country identification)
sample_vig = vig_df[vig_df["task_id"] == "vig_country_identification_normal"].iloc[0]
prompt_lines = sample_vig["prompt"].split("\n")
print("\n".join(prompt_lines[:35]))
print(f"\n... [{len(prompt_lines)} lines total]")
print(f"\nFirst 10 expected answers: {sample_vig['answers'][:10]}")

## 3. Expected Discrimination Patterns

What we expect to see when models are evaluated:

In [ ]:
# Hypothetical model performance curves — what we expect to measure
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: SIN accuracy curves (hypothetical)
ratios = [1, 5, 10, 25, 50, 100]
hypothetical_models = {
    "Strong model":    [1.0, 1.0, 0.96, 0.88, 0.72, 0.48],
    "Medium model":    [1.0, 0.96, 0.84, 0.64, 0.40, 0.20],
    "Weak model":      [0.96, 0.80, 0.56, 0.32, 0.16, 0.08],
}
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
for (name, accs), color in zip(hypothetical_models.items(), colors):
    axes[0].plot(ratios, accs, "o-", label=name, color=color, linewidth=2)
axes[0].axhline(y=0.8, color="gray", linestyle="--", alpha=0.7, label="80% threshold")
axes[0].set_xscale("log")
axes[0].set_xlabel("Noise ratio (log scale)")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Signal-in-Noise: Expected Accuracy Curves")
axes[0].set_xticks(ratios)
axes[0].set_xticklabels(ratios)
axes[0].legend(fontsize=9)
axes[0].set_ylim(-0.05, 1.05)
axes[0].annotate("← Attention\n    threshold", xy=(10, 0.8), fontsize=8, color="gray")

# Right: Vigilance decay curves (hypothetical)
positions = np.arange(1, 101)
np.random.seed(42)
def make_decay(base, onset, rate, noise=0.03):
    acc = np.ones(100) * base
    for i in range(onset, 100):
        acc[i] = base - rate * (i - onset) / 100
    return np.clip(acc + np.random.normal(0, noise, 100), 0, 1)

for (name, params), color in zip({
    "Strong model": (0.99, 70, 0.15),
    "Medium model": (0.97, 40, 0.25),
    "Weak model":   (0.95, 15, 0.35),
}.items(), colors):
    curve = make_decay(*params)
    # Smooth for display
    window = 5
    smoothed = np.convolve(curve, np.ones(window)/window, mode="valid")
    axes[1].plot(np.arange(window//2+1, 101-window//2), smoothed, label=name, color=color, linewidth=2)

axes[1].axhline(y=0.95, color="gray", linestyle="--", alpha=0.7, label="95% threshold")
axes[1].set_xlabel("Sub-task position")
axes[1].set_ylabel("Rolling accuracy (window=5)")
axes[1].set_title("Vigilance: Expected Decay Curves")
axes[1].legend(fontsize=9)
axes[1].set_ylim(0.5, 1.02)
axes[1].annotate("← Decay onset", xy=(40, 0.95), fontsize=8, color="gray")

plt.tight_layout()
plt.show()

## 4. Run Evaluation (requires kaggle-benchmarks SDK)

This section calls LLMs via the Kaggle model proxy. On Kaggle notebooks the SDK is pre-installed.
On Colab/local, install first: `pip install -e ./kaggle-benchmarks`

In [ ]:
import re

try:
    import kaggle_benchmarks as kbench
    HAS_SDK = True
    print(f"kaggle_benchmarks SDK loaded")
except ImportError:
    HAS_SDK = False
    print("SDK not available — evaluation cells will be skipped."
          "\nThe exploration sections above still work fine.")

In [ ]:
# ---- Helper functions (inline so the notebook is self-contained) ----

def parse_numbered_answers(response: str, count: int) -> list[str]:
    """Parse numbered answers from LLM response."""
    lines = response.strip().split("\n")
    answers = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        cleaned = re.sub(r"^\d+[\.\)\:]\s*", "", line)
        if cleaned:
            answers.append(cleaned)
    while len(answers) < count:
        answers.append("")
    return answers[:count]

def check_answer(expected: str, actual: str) -> bool:
    """Flexible answer matching."""
    exp = expected.lower().strip()
    act = actual.lower().strip()
    if exp == act or exp in act:
        return True
    exp_core = re.sub(r"^(approximately|about|roughly|around)\s+", "", exp)
    if exp_core in act:
        return True
    exp_d = re.sub(r"[,\s]", "", exp)
    act_d = re.sub(r"[,\s]", "", act)
    return exp_d == act_d and exp_d != ""

### 4a. Signal-in-Noise Evaluation

Runs a subset of SIN items against available models. Adjust `MODELS` and `MAX_ITEMS_PER_CONFIG` below.

In [ ]:
# Configuration — adjust these
MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4",
]
MAX_ITEMS_PER_CONFIG = 3  # items per (noise_type, noise_ratio) combo. Set higher for more signal.
SKIP_HIGH_RATIOS = True   # Skip 50:1 and 100:1 to save tokens (very long prompts)

In [ ]:
%%time

if not HAS_SDK:
    print("Skipping — SDK not available")
else:
    # Subsample the dataset
    sin_eval = sin_df.copy()
    if SKIP_HIGH_RATIOS:
        sin_eval = sin_eval[sin_eval["noise_ratio"] <= 25]

    # Take MAX_ITEMS_PER_CONFIG per (noise_type, noise_ratio)
    sin_eval = sin_eval.groupby(["noise_type", "noise_ratio"]).head(MAX_ITEMS_PER_CONFIG).reset_index(drop=True)
    print(f"Evaluating {len(sin_eval)} SIN items × {len(MODELS)} models = {len(sin_eval) * len(MODELS)} LLM calls")

    sin_results = []
    for model_name in MODELS:
        llm = kbench.llms[model_name]
        for _, row in sin_eval.iterrows():
            try:
                response = llm.prompt(row["prompt"])
                expected = row["answers"]
                parsed = parse_numbered_answers(response, row["num_questions"])
                correct = sum(1 for e, a in zip(expected, parsed) if check_answer(e, a))
                sin_results.append({
                    "model": model_name,
                    "task_id": row["task_id"],
                    "passage_id": row["passage_id"],
                    "domain": row["domain"],
                    "noise_type": row["noise_type"],
                    "noise_ratio": row["noise_ratio"],
                    "correct": correct,
                    "total": row["num_questions"],
                    "accuracy": correct / row["num_questions"],
                    "parsed_answers": parsed,
                    "raw_response": response[:500],
                })
            except Exception as e:
                print(f"  Error on {row['task_id']} with {model_name}: {e}")
                sin_results.append({
                    "model": model_name,
                    "task_id": row["task_id"],
                    "passage_id": row["passage_id"],
                    "domain": row["domain"],
                    "noise_type": row["noise_type"],
                    "noise_ratio": row["noise_ratio"],
                    "correct": 0,
                    "total": row["num_questions"],
                    "accuracy": 0.0,
                    "parsed_answers": [],
                    "raw_response": str(e),
                })
        print(f"  {model_name} done.")

    sin_results_df = pd.DataFrame(sin_results)
    print(f"\nCollected {len(sin_results_df)} results.")

In [ ]:
if HAS_SDK and len(sin_results_df):
    # Accuracy by noise ratio per model (averaged across noise types)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

    for ax, ntype in zip(axes, ["unrelated", "related", "adversarial"]):
        subset = sin_results_df[sin_results_df["noise_type"] == ntype]
        for model_name in MODELS:
            model_data = subset[subset["model"] == model_name]
            curve = model_data.groupby("noise_ratio")["accuracy"].mean()
            ax.plot(curve.index, curve.values, "o-", label=model_name.split("/")[-1], linewidth=2)
        ax.axhline(y=0.8, color="gray", linestyle="--", alpha=0.5)
        ax.set_xscale("log")
        ax.set_xlabel("Noise ratio")
        ax.set_title(f"Noise type: {ntype}")
        ax.set_xticks(sorted(subset["noise_ratio"].unique()))
        ax.set_xticklabels(sorted(subset["noise_ratio"].unique()))
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=8)

    axes[0].set_ylabel("Accuracy (5 questions)")
    fig.suptitle("Signal-in-Noise: Accuracy vs Noise Ratio", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    # Attention thresholds
    print("\nAttention Thresholds (highest ratio with ≥80% accuracy):")
    for model_name in MODELS:
        for ntype in ["unrelated", "related", "adversarial"]:
            subset = sin_results_df[(sin_results_df["model"] == model_name) & (sin_results_df["noise_type"] == ntype)]
            curve = subset.groupby("noise_ratio")["accuracy"].mean()
            threshold = 0
            for ratio in sorted(curve.index):
                if curve[ratio] >= 0.8:
                    threshold = ratio
            print(f"  {model_name.split('/')[-1]:20s} | {ntype:12s} | threshold = {threshold}:1")

### 4b. Vigilance Decrement Evaluation

Runs all 6 vigilance items (3 task types × normal/oddball) and analyzes accuracy by position.

In [ ]:
%%time

if not HAS_SDK:
    print("Skipping — SDK not available")
else:
    vig_results = []
    # Only evaluate normal variants for position analysis
    vig_normal = vig_df[vig_df["variant"] == "normal"]

    for model_name in MODELS:
        llm = kbench.llms[model_name]
        for _, row in vig_normal.iterrows():
            try:
                response = llm.prompt(row["prompt"])
                expected = row["answers"]
                parsed = parse_numbered_answers(response, row["num_subtasks"])
                position_correct = [
                    check_answer(e, a) for e, a in zip(expected, parsed)
                ]
                correct = sum(position_correct)
                vig_results.append({
                    "model": model_name,
                    "task_id": row["task_id"],
                    "task_type": row["task_type"],
                    "variant": row["variant"],
                    "correct": correct,
                    "total": row["num_subtasks"],
                    "accuracy": correct / row["num_subtasks"],
                    "position_correct": position_correct,
                })
                print(f"  {model_name.split('/')[-1]} | {row['task_type']:25s} | {correct}/{row['num_subtasks']}")
            except Exception as e:
                print(f"  Error: {model_name} on {row['task_id']}: {e}")
        print()

    # Also run oddball variants
    vig_oddball = vig_df[vig_df["variant"] == "oddball"]
    oddball_results = []
    for model_name in MODELS:
        llm = kbench.llms[model_name]
        for _, row in vig_oddball.iterrows():
            try:
                response = llm.prompt(row["prompt"])
                expected = row["answers"]
                parsed = parse_numbered_answers(response, row["num_subtasks"])
                position_correct = [
                    check_answer(e, a) for e, a in zip(expected, parsed)
                ]
                oddball_pos = row["oddball_position"]
                oddball_hit = position_correct[oddball_pos] if oddball_pos < len(position_correct) else False
                oddball_results.append({
                    "model": model_name,
                    "task_type": row["task_type"],
                    "oddball_position": oddball_pos,
                    "oddball_correct": oddball_hit,
                    "overall_accuracy": sum(position_correct) / len(position_correct),
                })
                print(f"  {model_name.split('/')[-1]} | {row['task_type']:25s} | oddball@{oddball_pos}: {'HIT' if oddball_hit else 'MISS'}")
            except Exception as e:
                print(f"  Error: {e}")

    vig_results_df = pd.DataFrame(vig_results)
    oddball_results_df = pd.DataFrame(oddball_results)
    print(f"\nCollected {len(vig_results_df)} vigilance + {len(oddball_results_df)} oddball results.")

In [ ]:
if HAS_SDK and len(vig_results_df):
    # Position-level accuracy curves
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    task_types = vig_results_df["task_type"].unique()

    for ax, ttype in zip(axes, task_types):
        for model_name in MODELS:
            row = vig_results_df[(vig_results_df["model"] == model_name) & (vig_results_df["task_type"] == ttype)]
            if len(row) == 0:
                continue
            pos_correct = np.array(row.iloc[0]["position_correct"], dtype=float)
            # Rolling average
            window = 10
            rolling = np.convolve(pos_correct, np.ones(window)/window, mode="valid")
            x = np.arange(window//2 + 1, len(pos_correct) - window//2 + 1)
            ax.plot(x, rolling, label=model_name.split("/")[-1], linewidth=2)

        ax.axhline(y=0.95, color="gray", linestyle="--", alpha=0.5)
        ax.set_xlabel("Sub-task position")
        ax.set_title(ttype.replace("_", " ").title())
        ax.set_ylim(0.4, 1.02)
        ax.legend(fontsize=8)

    axes[0].set_ylabel("Rolling accuracy (window=10)")
    fig.suptitle("Vigilance Decrement: Accuracy by Position", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    # Oddball detection
    if len(oddball_results_df):
        print("\nOddball Detection:")
        for _, row in oddball_results_df.iterrows():
            print(f"  {row['model'].split('/')[-1]:20s} | {row['task_type']:25s} | "
                  f"oddball@{row['oddball_position']}: {'DETECTED' if row['oddball_correct'] else 'MISSED'} | "
                  f"overall: {row['overall_accuracy']:.0%}")

### 4c. Save Results

In [ ]:
if HAS_SDK and len(sin_results_df):
    RESULTS_DIR = os.path.join(REPO_ROOT, "results")
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # Save SIN results (drop raw response for size)
    sin_save = sin_results_df.drop(columns=["raw_response", "parsed_answers"], errors="ignore")
    sin_save.to_json(os.path.join(RESULTS_DIR, "sin_results.json"), orient="records", indent=2)

    # Save vigilance results
    vig_save = vig_results_df.copy()
    vig_save.to_json(os.path.join(RESULTS_DIR, "vig_results.json"), orient="records", indent=2)

    if len(oddball_results_df):
        oddball_results_df.to_json(os.path.join(RESULTS_DIR, "oddball_results.json"), orient="records", indent=2)

    print(f"Results saved to {RESULTS_DIR}/")
else:
    print("No results to save.")